## Ejercicio 1
¿Qué algoritmo de entrenamiento de regresión lineal puedes usar si tienes un conjunto de entrenamiento con millones de características?

---

Se utilizaría una regularización 'L1' la cual ayuda a encontrar aquellas caracteristicas que son utiles para la regresión de manera agresiva llevando caracteristicas poco importantes a pesos 0 pudiendo seleccionar acaracteristicas

## Ejercicio 2
Supón que las características en tu conjunto de entrenamiento tienen escalas muy diferentes. ¿Qué algoritmos podrían sufrir por esto, y cómo? ¿Qué puedes hacer al respecto?

SDG podría sufrir, ya que los gradientes al existir caracteristicas con escalas muy diferentes, una forma de abatir esto es escalando las caracteristicas con escaladores de scikit-learn como StandardScaler()

## Ejercicio 3
¿Puede el descenso de gradiente quedarse atascado en un mínimo local al entrenar un modelo de regresión logística?

En un principio no por que la función de minimización es entropía cruzada la cual es convexa, entonces el gradiente puede encontrar ese minimo global 

## Ejercicio 4
¿Todos los algoritmos de descenso de gradiente llevan al mismo modelo, siempre que los dejes correr el tiempo suficiente?

En un principio debería pero también influye el schedule del learning rate,, la regularización si es que se utiliza pero al ser concava la funcion a minimizar es verdad que si debería de llegar

## Ejercicio 5
Supón que usas batch gradient descent y graficas el error de validación en cada época. Si notas que el error de validación sube de forma consistente, ¿qué está ocurriendo probablemente? ¿Cómo puedes solucionarlo?

Lo más probable es que el modelo no generaliza de manera correcta, se esta haciendo un fenomeno de overfitting

Esto se puede combatir añadiendo más datos para que al modelo encuentre patrones para poder generalizar mejor, usar regularización para evitar el overfitting o por ultimo reducir la complejidad del modelo, por ejemplo, buscar algoritmos menos complejos, reducir iteraciones intentar con un modelo más simple etc.

## Ejercicio 6
¿Es buena idea detener mini-batch gradient descent inmediatamente cuando el error de validación sube?

No, ya que el mini-batch se distingue por ser estocastico, así que es posible que pueda encontrar minimizar la validación en las siguientes iteraciones, se recomienda usar en este caso el early-stop para que dadas n cantidad de iteraciones sin mejora se detenga el entrenamiento y tome el punto con la mejor validación hsitorica

## Ejercicio 7
¿Qué algoritmo de descenso de gradiente (entre los que discutimos) llegará más rápido a la vecindad de la solución óptima? ¿Cuál convergerá realmente? ¿Cómo puedes hacer que los demás también converjan?

Probablemente el full-batch ya que da pasos certeros a diferencia del mini batch que da pasos estocasticos, el min bacth lo que tiene es que tambipén lleca así como el estocastico el mini pues tiene a ser mas loco

Para lograr que converjan todo, se puede hacer de un scheduler de learning rate ajjkustado, verificar las perdida en las validaciónes 

## Ejercicio 8
Supón que usas regresión polinómica. Graficas las curvas de aprendizaje y notas una gran brecha entre el error de entrenamiento y el de validación. ¿Qué está pasando? ¿Cuáles son tres formas de resolverlo?

A esto se le conoce como underfitting, el modelo no logra aprender de manera eficiente del conjunto de datos, aquí las 3 soluciones son:

- Aumentar la complejidad, probar un modelo más complejo para ver si es capaz de aprender mejor
- Mejorar la calidad del conjunto de datos, reducirlo cando naquellas instancias que no benefician el aprendizaje como outliers
- Escoger mejor los predictores

## Ejercicio 9
Supón que usas ridge regression y notas que el error de entrenamiento y el de validación son casi iguales y bastante altos. ¿Dirías que el modelo sufre de alto sesgo o alta varianza? ¿Deberías aumentar el hiperparámetro de regularización α o reducirlo?

sería alto sesgo, el modelo no es capaz de aprender de manera eficiente del conjunto de datos, para esto se recomienda reducir el hiperparámetro de regularización para que el modelo pueda aprender mejor del conjunto de datos

## Ejercicio 10
¿Por qué querrías usar:
a. Ridge regression en lugar de regresión lineal simple (sin regularización)?
b. Lasso en lugar de ridge regression?
c. Elastic net en lugar de lasso regression?

Se utiliza Ridge por defecto cuando se quieres usar una regularización en un caso donde quieres suavizar un modelo demasiado complejo para hacer que variables menos contribuyentes tomen menor peso.

Se usa Lasso por enecima de una regresión simple en el caso en que tengas muchas caracteristicas y necesites quedarte con las más importantes, es una regularización más agresiva que penaliza pesoss poco importantes

Elastic Net se usa en el caso de que quieras regularizar un poco de los 2 mundos, igual par suavizar

## Ejercicio 11
Supón que quieres clasificar imágenes como outdoor/indoor y daytime/nighttime. ¿Deberías implementar dos clasificadores de regresión logística o un único clasificador de regresión softmax?

Como son binarios lo mejor es usar 2 clasificadores ya que son caracteristicas a clasificar diferentes

## Ejercicio 12
Implementa batch gradient descent con early stopping para regresión softmax sin usar Scikit-Learn, solo NumPy. Úsalo en una tarea de clasificación como el dataset iris.

In [40]:
import numpy as np

def batch_gradient_descend(X, y, alpha, w_0, n_iter, early_stop, tol=1e-8):
    m = X.shape[0]
    w = w_0.copy()
    count = 0
    best_loss = np.inf
    best_w = None

    for i in range(n_iter):
        z = X @ w
        z -= np.max(z, axis=1, keepdims=True)  # Para estabilidad numérica

        exp_z = np.exp(z)
        sum_exp_z = np.sum(exp_z,  axis=1, keepdims=True)

        p = exp_z / sum_exp_z

        grad = (1/m) * X.T @ (p - y)
        w -= alpha * grad

        n_loss = - (1/m) * np.sum(np.sum(y * np.log(p + 1e-15)))

        print(f"Iteration {i+1}/{n_iter}, Loss: {n_loss:.4f}")

        if best_loss > n_loss:
            best_loss = n_loss
            best_w = w.copy()
            count = 0
        else:
            count += 1

        if count >= early_stop and abs(best_loss - n_loss) < tol:
            print(f"Early stopping at iteration {i+1} with loss {best_loss:.8f}")
            break

 
    return best_w

Probamos con un problema simple, este bgd con early stopping para regresión softmax

In [41]:
X = np.array([[1, 2], [3, 4], [5, 6]])
y = np.array([[0, 1], [1, 0], [0, 1]])
alpha = 0.01
w_0 = np.zeros((X.shape[1], y.shape[1]))
n_iter = 100000
early_stop = 15
best_weights = batch_gradient_descend(X, y, alpha, w_0, n_iter, early_stop)
print("Best weights:\n", best_weights)

Iteration 1/100000, Loss: 0.6931
Iteration 2/100000, Loss: 0.6803
Iteration 3/100000, Loss: 0.6710
Iteration 4/100000, Loss: 0.6644
Iteration 5/100000, Loss: 0.6595
Iteration 6/100000, Loss: 0.6560
Iteration 7/100000, Loss: 0.6534
Iteration 8/100000, Loss: 0.6515
Iteration 9/100000, Loss: 0.6501
Iteration 10/100000, Loss: 0.6490
Iteration 11/100000, Loss: 0.6483
Iteration 12/100000, Loss: 0.6477
Iteration 13/100000, Loss: 0.6472
Iteration 14/100000, Loss: 0.6469
Iteration 15/100000, Loss: 0.6467
Iteration 16/100000, Loss: 0.6465
Iteration 17/100000, Loss: 0.6464
Iteration 18/100000, Loss: 0.6462
Iteration 19/100000, Loss: 0.6462
Iteration 20/100000, Loss: 0.6461
Iteration 21/100000, Loss: 0.6460
Iteration 22/100000, Loss: 0.6460
Iteration 23/100000, Loss: 0.6460
Iteration 24/100000, Loss: 0.6459
Iteration 25/100000, Loss: 0.6459
Iteration 26/100000, Loss: 0.6459
Iteration 27/100000, Loss: 0.6459
Iteration 28/100000, Loss: 0.6459
Iteration 29/100000, Loss: 0.6459
Iteration 30/100000, Lo

Vemos que funciona de manera correcto el algoritmo junto con el early stopping.

In [42]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
list(iris)

['data',
 'target',
 'frame',
 'target_names',
 'DESCR',
 'feature_names',
 'filename',
 'data_module']

In [43]:
iris.data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [44]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [45]:
X = iris.data.values
y = iris.target.values

# agregar bias
X = np.hstack([np.ones((X.shape[0], 1)), X])

In [46]:
# Convertir y a one-hot encoding
def ohe(y):
    n_classes = np.max(y) + 1
    return np.eye(n_classes)[y]
y = ohe(y)
# print de y 5 valores random
print("y (one-hot encoding) sample:\n", y[np.random.choice(y.shape[0], 5, replace=False)])

y (one-hot encoding) sample:
 [[0. 1. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]


In [47]:
# Dividir el dataset en entrenamiento y prueba solo numpy
def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)
    split_idx = int(X.shape[0] * (1 - test_size))
    train_indices = indices[:split_idx]
    test_indices = indices[split_idx:]
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (120, 5)
y_train shape: (120, 3)
X_test shape: (30, 5)
y_test shape: (30, 3)


In [48]:
n_iter = 10000
w_0 = np.zeros((X_train.shape[1], y_train.shape[1]))
alpha = 0.1
best_weights = batch_gradient_descend(X_train, y_train, alpha, w_0, n_iter, early_stop)
print("Best weights:\n", best_weights)

Iteration 1/10000, Loss: 1.0986
Iteration 2/10000, Loss: 1.0278
Iteration 3/10000, Loss: 0.9670
Iteration 4/10000, Loss: 0.9158
Iteration 5/10000, Loss: 0.8747
Iteration 6/10000, Loss: 0.8407
Iteration 7/10000, Loss: 0.8161
Iteration 8/10000, Loss: 0.7990
Iteration 9/10000, Loss: 0.7873
Iteration 10/10000, Loss: 0.7865
Iteration 11/10000, Loss: 0.7768
Iteration 12/10000, Loss: 0.7854
Iteration 13/10000, Loss: 0.7642
Iteration 14/10000, Loss: 0.7761
Iteration 15/10000, Loss: 0.7440
Iteration 16/10000, Loss: 0.7589
Iteration 17/10000, Loss: 0.7224
Iteration 18/10000, Loss: 0.7411
Iteration 19/10000, Loss: 0.7029
Iteration 20/10000, Loss: 0.7249
Iteration 21/10000, Loss: 0.6859
Iteration 22/10000, Loss: 0.7105
Iteration 23/10000, Loss: 0.6708
Iteration 24/10000, Loss: 0.6975
Iteration 25/10000, Loss: 0.6574
Iteration 26/10000, Loss: 0.6857
Iteration 27/10000, Loss: 0.6452
Iteration 28/10000, Loss: 0.6749
Iteration 29/10000, Loss: 0.6341
Iteration 30/10000, Loss: 0.6648
Iteration 31/10000,

Iteration 1839/10000, Loss: 0.1070
Iteration 1840/10000, Loss: 0.1069
Iteration 1841/10000, Loss: 0.1069
Iteration 1842/10000, Loss: 0.1069
Iteration 1843/10000, Loss: 0.1069
Iteration 1844/10000, Loss: 0.1069
Iteration 1845/10000, Loss: 0.1068
Iteration 1846/10000, Loss: 0.1068
Iteration 1847/10000, Loss: 0.1068
Iteration 1848/10000, Loss: 0.1068
Iteration 1849/10000, Loss: 0.1068
Iteration 1850/10000, Loss: 0.1067
Iteration 1851/10000, Loss: 0.1067
Iteration 1852/10000, Loss: 0.1067
Iteration 1853/10000, Loss: 0.1067
Iteration 1854/10000, Loss: 0.1067
Iteration 1855/10000, Loss: 0.1066
Iteration 1856/10000, Loss: 0.1066
Iteration 1857/10000, Loss: 0.1066
Iteration 1858/10000, Loss: 0.1066
Iteration 1859/10000, Loss: 0.1066
Iteration 1860/10000, Loss: 0.1065
Iteration 1861/10000, Loss: 0.1065
Iteration 1862/10000, Loss: 0.1065
Iteration 1863/10000, Loss: 0.1065
Iteration 1864/10000, Loss: 0.1065
Iteration 1865/10000, Loss: 0.1064
Iteration 1866/10000, Loss: 0.1064
Iteration 1867/10000

In [49]:
def predict(X, w, class_names):
    z = X @ w
    z -= np.max(z, axis=1, keepdims=True)  # Para estabilidad numérica
    exp_z = np.exp(z)
    sum_exp_z = np.sum(exp_z, axis=1, keepdims=True)
    p = exp_z / sum_exp_z

    return np.argmax(p, axis=1)

class_names = iris.target_names
y_pred = predict(X_test, best_weights, class_names)
print("Predicted classes:\n", y_pred)

Predicted classes:
 [1 0 1 1 0 1 2 2 0 1 2 1 0 2 0 1 2 2 1 2 1 1 2 2 0 1 2 0 1 2]


In [50]:
# calcula el accuracy
accuracy = np.mean(y_pred == np.argmax(y_test, axis=1))
print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.9667
